# Named Entity Recognition (NER)

This Notebook is intended to show the process of training and testing the NER model.

In [ ]:
from __future__ import print_function
from utils.ner_functions import train_loop, val_loop, adjust_learning_rate, load_conll_data, load_pretrained_embeddings, save_mappings, load_sentences, update_tag_scheme, split_metrics, plot_label_distribution, flatten_tags
from Models.BiLSTM_CRF import BiLSTM_CRF
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter
import pandas as pd
from collections import Counter, OrderedDict
import os

import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

plt.rcParams['figure.dpi'] = 80

## Parameters for the Model

In [3]:
parameters = OrderedDict()
parameters['train'] = "../data/eng.train" #Path to train file
parameters['dev'] = "../data/eng.testa" #Path to test file
parameters['test'] = "../data/eng.testb" #Path to dev file
parameters['tag_scheme'] = "BIOES" 
parameters['lower'] = True 
parameters['zeros'] =  True # Boolean variable to control replacement of  all digits by 0 
parameters['char_dim'] = 30 
parameters['word_dim'] = 100 
parameters['word_lstm_dim'] = 200 
parameters['char_lstm_dim'] = 25 # or 50
parameters['word_bidirect'] = True
parameters['embedding_path'] = "../data/glove.6B.100d.txt" #Location of pretrained embeddings
parameters['all_emb'] = 1 #Load all embeddings
parameters['crf'] =1 
parameters['dropout'] = 0.5 
parameters['epoch'] =  50 
parameters['weights'] = "" #path to Pretrained for from a previous run
parameters['name'] = "self-trained-model" 
parameters['gradient_clip']=5.0
parameters['char_mode']="CNN"
models_path = "./models/" 

#GPU
parameters['use_gpu'] = torch.cuda.is_available() 
use_gpu = parameters['use_gpu']
parameters['reload'] = "../models/pre-trained-model" 

#paths to stored mapping file
mapping_file = '../data/mapping.pkl'
#To stored model
name = parameters['name']
model_name = models_path + name #get_name(parameters)

if not os.path.exists(models_path):
    os.makedirs(models_path)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)
print(torch.cuda.current_device())


cuda
0


# Data Processing

In [ ]:
train_data, dev_data, test_data, (word_to_id, char_to_id, tag_to_id) = load_conll_data(
    train_path=parameters['train'],
    dev_path=parameters['dev'],
    test_path=parameters['test'],
    params=parameters,
    debug=True
)

train sentences: 14041
test sentences: 3453
dev sentences: 3250
train_sentences[0]:  [['EU', 'NNP', 'I-NP', 'I-ORG'], ['rejects', 'VBZ', 'I-VP', 'O'], ['German', 'JJ', 'I-NP', 'I-MISC'], ['call', 'NN', 'I-NP', 'O'], ['to', 'TO', 'I-VP', 'O'], ['boycott', 'VB', 'I-VP', 'O'], ['British', 'JJ', 'I-NP', 'I-MISC'], ['lamb', 'NN', 'I-NP', 'O'], ['.', '.', 'O', 'O']]
Found 17493 unique words (203621 in total)
Found 75 unique characters
Found 19 unique named entity tags
{'S-ORG': 3836, 'O': 169578, 'S-MISC': 2580, 'B-PER': 4284, 'E-PER': 4284, 'S-LOC': 6099, 'B-ORG': 2485, 'E-ORG': 2485, 'I-PER': 244, 'S-PER': 2316, 'B-MISC': 858, 'I-MISC': 297, 'E-MISC': 858, 'I-ORG': 1219, 'B-LOC': 1041, 'E-LOC': 1041, 'I-LOC': 116, '<START>': -1, '<STOP>': -2}


## Data Analysis
Quick EDA to inspect class balance and split-level statistics before training. Some of the most important observations we made in this analysis are:
- All entity types are present in the three splits.
- The presence of O Tag (no entity) is as prevalent in all the splits (82-83%)
- The splits are completely balanced in terms of entity type. The percentage of a tag in any of the splits differs by less than 1%. There is no major distribution shift.
- Since there is a clear imbalance towards O-Tags, F1 is the right metric for this dataset.

In [ ]:

split_paths = {"train": parameters["train"], "val": parameters["dev"], "test": parameters["test"]}
split_sentences = {}
visual_representation = False 

for split_name, split_path in split_paths.items():
    sents = load_sentences(split_path, parameters["zeros"])
    update_tag_scheme(sents, parameters["tag_scheme"])
    split_sentences[split_name] = sents

summary_df = pd.DataFrame({name: split_metrics(sents) for name, sents in split_sentences.items()}).T
summary_df = summary_df[["sentences", "tokens", "avg_sentence_len", "entity_types", "pct_O", "pct_entity",]]

print("Split-level metrics")
display(summary_df.round(2))

label_counts = {name: Counter(flatten_tags(sents)) for name, sents in split_sentences.items()}
all_labels = sorted(set().union(*(counts.keys() for counts in label_counts.values())))

label_dist_df = pd.DataFrame(
    {
        split_name: [label_counts[split_name].get(label, 0) for label in all_labels]
        for split_name in split_sentences.keys()
    },
    index=all_labels,
)

label_dist_df["total"] = label_dist_df.sum(axis=1)
label_dist_df = label_dist_df.sort_values("total", ascending=False).drop(columns=["total"])
label_pct_df = 100.0 * label_dist_df.div(label_dist_df.sum(axis=0), axis=1)

if visual_representation:
    plot_label_distribution(label_dist_df, summary_df)
else:
    print("Raw label counts by split")
    display(label_dist_df)

    print("Label percentages by split")
    display(label_pct_df.round(3))


Split-level metrics


,sentences,tokens,avg_sentence_len,entity_types,pct_O,pct_entity
train,14041.0,203621.0,14.50,4.0,83.28,16.72
val,3250.0,51362.0,15.80,4.0,83.25,16.75
test,3453.0,46435.0,13.45,4.0,82.53,17.47


Raw label counts by split


,train,val,test
O,169578,42759,38323
S-LOC,6099,1603,1436
B-PER,4284,1234,1086
E-PER,4284,1234,1086
S-ORG,3836,891,1082
S-MISC,2580,665,525
E-ORG,2485,450,579
B-ORG,2485,450,579
S-PER,2316,608,531
I-ORG,1219,301,256


Label percentages by split


,train,val,test
O,83.281,83.250,82.530
S-LOC,2.995,3.121,3.092
B-PER,2.104,2.403,2.339
E-PER,2.104,2.403,2.339
S-ORG,1.884,1.735,2.330
S-MISC,1.267,1.295,1.131
E-ORG,1.220,0.876,1.247
B-ORG,1.220,0.876,1.247
S-PER,1.137,1.184,1.144
I-ORG,0.599,0.586,0.551


In [5]:
word_embeds = load_pretrained_embeddings(
    parameters["embedding_path"],
    word_to_id,
    parameters["word_dim"]
)

save_mappings(mapping_file, word_to_id, tag_to_id,
              char_to_id, parameters, word_embeds)

Loaded 15974 pretrained embeddings out of 17493.


{'word_to_id': {'<UNK>': 0,
  'the': 1,
  '.': 2,
  ',': 3,
  '0': 4,
  'of': 5,
  'in': 6,
  '00': 7,
  'to': 8,
  'a': 9,
  'and': 10,
  '(': 11,
  ')': 12,
  '"': 13,
  'on': 14,
  'said': 15,
  "'s": 16,
  'for': 17,
  '-': 18,
  'at': 19,
  'was': 20,
  '0-0': 21,
  '000': 22,
  '0000-00-00': 23,
  'with': 24,
  'that': 25,
  'he': 26,
  '0000': 27,
  'from': 28,
  'it': 29,
  'by': 30,
  'is': 31,
  ':': 32,
  'as': 33,
  '0.': 34,
  'had': 35,
  'has': 36,
  'his': 37,
  'but': 38,
  'an': 39,
  'not': 40,
  'were': 41,
  'be': 42,
  'after': 43,
  'have': 44,
  'first': 45,
  'new': 46,
  'who': 47,
  'will': 48,
  'they': 49,
  'two': 50,
  '00.00': 51,
  '0.00': 52,
  'u.s.': 53,
  'been': 54,
  '$': 55,
  '--': 56,
  'their': 57,
  'beat': 58,
  'are': 59,
  'which': 60,
  'would': 61,
  'this': 62,
  'up': 63,
  'its': 64,
  'year': 65,
  'i': 66,
  'last': 67,
  'percent': 68,
  'out': 69,
  'we': 70,
  'thursday': 71,
  'one': 72,
  'million': 73,
  'over': 74,
  '00th': 

# Models Implementation
### Global Parameters

In [6]:
learning_rate = 0.015
momentum = 0.9
number_of_epochs = parameters['epoch'] 
decay_rate = 0.05
gradient_clip = parameters['gradient_clip']
losses = [] 
loss = 0.0
best_dev_F = -1.0 # Current best F-1 Score on Dev Set
best_test_F = -1.0 # Current best F-1 Score on Test Set
best_train_F = -1.0 # Current best F-1 Score on Train Set
all_F = [[0, 0, 0]] 
eval_every = len(train_data) 
plot_every = 2000 
count = 0 

In [7]:
model = BiLSTM_CRF(
    vocab_size=len(word_to_id),
    tag_to_ix=tag_to_id,
    embedding_dim=parameters['word_dim'],
    hidden_dim=parameters['word_lstm_dim'],
    char_to_ix=char_to_id,
    char_lstm_dim=parameters['char_lstm_dim'], 
    use_gpu=use_gpu,
    device=device
).to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)


# Train & Evaluation

In this section we compare the different models we trained:
1. Train Loss: 0.0182 | Train F1: 0.9973 | Val Loss: 1.6385 | Val F1: 0.9006 | Test Loss: 3.0330 | Test F1: 0.8337
2. Train Loss: 0.0149 | Train F1: 0.9979 | Val Loss: 1.6815 | Val F1: 0.8993 | Test Loss: 2.9945 | Test F1: 0.8441
3. Test Loss: 1.9883 | Test F1: 0.8723


In [ ]:
model_name = f"ner_bilstm_crf_hd_{parameters['word_lstm_dim']}_lr_{learning_rate}"
train_writer = SummaryWriter(f"runs/{model_name}/train")
val_writer = SummaryWriter(f"runs/{model_name}/val")

for epoch in tqdm(range(parameters['epoch'])):
    train_loss, train_f1 = train_loop(model, train_data, optimizer, gradient_clip, device, train_writer, epoch)
    val_loss, val_f1 = val_loop(model, dev_data, device, val_writer, epoch)

    print(f"Epoch: {epoch} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")

    adjust_learning_rate(optimizer, lr=learning_rate/(1 + 0.05 * epoch))

torch.save(model.state_dict(), f"models/{model_name}.pth")


  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
model_path = Path("models") / "../../models/ner_models/best_ner_model.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

test_writer = SummaryWriter(f"runs/{model_name}/test")
test_loss, test_f1 = val_loop(model, test_data, device, test_writer, epoch=0)

print(f"Test Loss: {test_loss:.4f} | Test F1: {test_f1:.4f}")

Test Loss: 0.0187 | Test F1: 0.9973


# Model Usage Example

In this section we will show an example of the usage of the model.

- example_phrases = ["Donald Trump went to the USA this morning", "Jay is from India"]

In [ ]:
from utils.ner_functions import build_model, predict_ner, normalize_ner_output

model, word_to_id, char_to_id, tag_to_id, params, device = build_model(parameters)
phrases = ["Donald Trump went to the USA this morning", "NATO is an organization" ]

for phrase in phrases:
    predictions = predict_ner(phrase, model, word_to_id, char_to_id, tag_to_id, params, device)
    text = normalize_ner_output(predictions)
    print(text)

train sentences: 14041
test sentences: 3453
dev sentences: 3250
train_sentences[0]:  [['EU', 'NNP', 'I-NP', 'I-ORG'], ['rejects', 'VBZ', 'I-VP', 'O'], ['German', 'JJ', 'I-NP', 'I-MISC'], ['call', 'NN', 'I-NP', 'O'], ['to', 'TO', 'I-VP', 'O'], ['boycott', 'VB', 'I-VP', 'O'], ['British', 'JJ', 'I-NP', 'I-MISC'], ['lamb', 'NN', 'I-NP', 'O'], ['.', '.', 'O', 'O']]
Found 17493 unique words (203621 in total)
Found 75 unique characters
Found 19 unique named entity tags
{'S-ORG': 3836, 'O': 169578, 'S-MISC': 2580, 'B-PER': 4284, 'E-PER': 4284, 'S-LOC': 6099, 'B-ORG': 2485, 'E-ORG': 2485, 'I-PER': 244, 'S-PER': 2316, 'B-MISC': 858, 'I-MISC': 297, 'E-MISC': 858, 'I-ORG': 1219, 'B-LOC': 1041, 'E-LOC': 1041, 'I-LOC': 116, '<START>': -1, '<STOP>': -2}
Donald Trump (PERSON), USA (LOCATION)
NATO (ORGANIZATION)
